In [1]:
import pandas as pd

data = pd.read_csv('../Dataset/Scored_Cleaned_Data.csv')

In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

def evaluate_agglomerative_custom_logic(df, target_col='Risk_score', n_folds=5):
    X_df = df.drop(columns=["Timestamp", "Group"], errors='ignore')
    spend_values = df[target_col].values
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    overall_results = []

    print(f"{'K':<5} | {'Avg Silhouette':<15} | {'Custom Deviation':<20}")
    print("-" * 50)

    for k in range(3, 20):
        fold_silhouettes = []
        fold_custom_deviations = []
        
        for train_idx, val_idx in kf.split(X_df):
            X_fold = X_df.iloc[train_idx]
            spend_fold = spend_values[train_idx]
            
            model = AgglomerativeClustering(n_clusters=k, linkage='ward')
            labels = model.fit_predict(X_fold)
            
            # Silhouette
            sil = silhouette_score(X_fold, labels)
            fold_silhouettes.append(sil)
            
            # (sum(|M - means|^0.5))^2
            cluster_means = [spend_fold[labels == cid].mean() for cid in np.unique(labels)]
            c_means = np.array(cluster_means)
            M = np.mean(c_means)
            diffs = np.abs(M - c_means)
            custom_dev = (np.sum(diffs**0.5))**2
            fold_custom_deviations.append(custom_dev)

        avg_sil = np.mean(fold_silhouettes)
        avg_dev = np.mean(fold_custom_deviations)
        overall_results.append({'k': k, 'Silhouette': avg_sil, 'Custom_Deviation': avg_dev})
        print(f"{k:<5} | {avg_sil:<15.4f} | {avg_dev:<20.2f}")

    return pd.DataFrame(overall_results)

agg_custom_results = evaluate_agglomerative_custom_logic(data)

K     | Avg Silhouette  | Custom Deviation    
--------------------------------------------------
3     | 0.4433          | 33.90               
4     | 0.3844          | 83.46               
5     | 0.3520          | 298.31              
6     | 0.3557          | 695.47              
7     | 0.3700          | 1133.33             
8     | 0.3772          | 1739.09             
9     | 0.3773          | 2485.83             
10    | 0.3827          | 3249.85             
11    | 0.3868          | 4045.38             
12    | 0.3908          | 4839.57             
13    | 0.3952          | 5767.27             
14    | 0.4059          | 6650.11             
15    | 0.4213          | 7468.98             
16    | 0.4278          | 8231.87             
17    | 0.4373          | 9004.10             
18    | 0.4546          | 9439.66             
19    | 0.4641          | 10212.42            
